# Baseline: Dual-HMM

Two independent discrete HMMs, one for the payload and one for the IAT, as in IP traffic generator based on hidden Markov models (IcETRAN 2020). Payload and IAT are discretized separately into 50 bins each (k-means bin edges fitted on the training packets), each HMM with 4 hidden states is trained with Baum-Welch, and a generated symbol is mapped back to the center of its bin. Payload and IAT are generated independently of each other.

Run by `scripts/run_matrix.py` after the proposed model of the same dataset and seed, whose `split.json` and `test.csv` it reuses. The packets are cleaned with the same function as for the proposed model, and the run stops if its test flows differ from `test.csv`.

In [ ]:
SEED         = 0
DATASET      = 'facebook_video'
RAW_PATH_STR = 'data/facebook_video.csv'
OUT_DIR      = 'runs/facebook_video/seed0'
MODEL_KEY    = 'dualhmm'
TEST_SIZE    = 0.3              # not used: the split is read from split.json

In [ ]:
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from hmmlearn import hmm
from sklearn.preprocessing import KBinsDiscretizer
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, load_split_checked, postprocess_synthetic

ART = Path(OUT_DIR) / 'artifacts' / MODEL_KEY
ART.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
random.seed(SEED)

K_STATES_SIZE = 4       # hidden states of the payload HMM
K_STATES_TIME = 4       # hidden states of the IAT HMM
N_ITER_HMM    = 200     # maximum Baum-Welch iterations
M_VOCAB_SIZE  = 50      # payload bins (symbols)
M_VOCAB_TIME  = 50      # IAT bins (symbols)

In [ ]:
df = load_packets(RAW_PATH_STR)
train_flows, test_flows = load_split_checked(df, OUT_DIR)
train_mask = df['flow_id'].isin(train_flows)

# payload and IAT are discretized without a log transform
discretizer_size = KBinsDiscretizer(n_bins=M_VOCAB_SIZE, encode='ordinal', strategy='kmeans')
discretizer_time = KBinsDiscretizer(n_bins=M_VOCAB_TIME, encode='ordinal', strategy='kmeans')
discretizer_size.fit(df.loc[train_mask, ['payload_length']])
discretizer_time.fit(df.loc[train_mask, ['time_diff']])
df['size_bin'] = discretizer_size.transform(df[['payload_length']]).astype(int)
df['time_bin'] = discretizer_time.transform(df[['time_diff']]).astype(int)


def build_hmm_sequences(frame, col):
    seqs, lens = [], []
    for _, grp in frame.groupby('flow_id'):
        arr = grp[[col]].to_numpy(dtype=int)
        seqs.append(arr)
        lens.append(len(arr))
    return np.vstack(seqs), lens


train_df = df[train_mask]
X_size, len_size = build_hmm_sequences(train_df, 'size_bin')
X_time, len_time = build_hmm_sequences(train_df, 'time_bin')
hmm_size = hmm.CategoricalHMM(n_components=K_STATES_SIZE, n_iter=N_ITER_HMM, random_state=SEED, verbose=False)
hmm_size.fit(X_size, len_size)
hmm_time = hmm.CategoricalHMM(n_components=K_STATES_TIME, n_iter=N_ITER_HMM, random_state=SEED, verbose=False)
hmm_time.fit(X_time, len_time)
joblib.dump(hmm_size, ART / 'dualhmm_hmm_size.pkl')
joblib.dump(hmm_time, ART / 'dualhmm_hmm_time.pkl')

In [ ]:
# each HMM draws from its own random generator (created by fit from random_state), which advances
# from flow to flow; a symbol is mapped back to the center of its bin
records = []
for fid in tqdm(test_flows, desc='flows'):
    L = int(df.loc[df['flow_id'] == fid].shape[0])
    if L == 0:
        continue
    size_bins, _ = hmm_size.sample(L)
    time_bins, _ = hmm_time.sample(L)
    payloads = np.clip(discretizer_size.inverse_transform(size_bins.reshape(-1, 1)).flatten(), 0.0, 1500.0)
    dts = np.maximum(discretizer_time.inverse_transform(time_bins.reshape(-1, 1)).flatten(), 0.0)
    for i in range(L):
        records.append([fid, round(payloads[i]), float(dts[i])])
synth_df = pd.DataFrame(records, columns=['flow_id', 'payload_length', 'time_diff'])

joblib.dump(discretizer_size, ART / 'dualhmm_discretizer_size.pkl')
joblib.dump(discretizer_time, ART / 'dualhmm_discretizer_time.pkl')

synth_df = postprocess_synthetic(synth_df)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'saved {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')